# Notebook 4 — Stock Selection

For each confirmed anomaly window, this notebook ranks constituent stocks across all 6 factors:

| # | Factor | When to run |
|---|---|---|
| 1 | Participation rate | Pre-computed, run once |
| 2 | Window-specific beta | Pre-computed, run once |
| 3 | Liquidity | Pre-computed, run once |
| 4 | Events inside window | Check ~1 week before window opens |
| 5 | Relative strength at entry | Check on entry day |
| 6 | Fundamental quality floor | Check ~1 week before window opens |

**Output:** A ranked candidate list per anomaly window saved to `output/stock_selection.xlsx`

In [ ]:
# ── Install dependencies (run once) ──────────────────────────────────────────
# !uv add yfinance pandas numpy openpyxl requests tqdm

In [4]:
# ── Imports ───────────────────────────────────────────────────────────────────
import yfinance as yf
import pandas as pd
import numpy as np
import os
import warnings
from datetime import datetime, timedelta
from tqdm.notebook import tqdm
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.formatting.rule import ColorScaleRule
warnings.filterwarnings('ignore')
from nselib import indices

print('Libraries loaded.')

Libraries loaded.


In [5]:
# ── Configuration ─────────────────────────────────────────────────────────────
# Define your confirmed anomaly windows here.
# Add or remove entries as your research evolves.

ANOMALY_WINDOWS = [
    {
        'name'        : 'FMCG Mar-May Rally',
        'sector'      : 'Nifty FMCG',
        'start_month' : 3,
        'start_day'   : 26,
        'window_days' : 50,
    },
    {
        'name'        : 'Pharma Aug-Sep Rally',
        'sector'      : 'Nifty Pharma',
        'start_month' : 8,
        'start_day'   : 29,
        'window_days' : 10,
    },
    {
        'name'        : 'Energy Aug Mid Rally',
        'sector'      : 'Nifty Energy',
        'start_month' : 8,
        'start_day'   : 12,
        'window_days' : 17,
    },
    {
        'name'        : 'Pharma Aug Extended',
        'sector'      : 'Nifty Pharma',
        'start_month' : 8,
        'start_day'   : 25,
        'window_days' : 14,
    },
]

# Constituent stocks per sector
# These are the main index constituents — update if NSE changes composition
SECTOR_STOCKS = {
    'Nifty FMCG': [
        'HINDUNILVR.NS', 'ITC.NS', 'NESTLEIND.NS', 'BRITANNIA.NS',
        'DABUR.NS', 'MARICO.NS', 'GODREJCP.NS', 'COLPAL.NS',
        'EMAMILTD.NS', 'TATACONSUM.NS', 'UBL.NS', 'MCDOWELL-N.NS',
    ],
    'Nifty Pharma': [
        'SUNPHARMA.NS', 'DRREDDY.NS', 'CIPLA.NS', 'DIVISLAB.NS',
        'BIOCON.NS', 'AUROPHARMA.NS', 'ALKEM.NS', 'TORNTPHARM.NS',
        'LUPIN.NS', 'ABBOTINDIA.NS', 'IPCALAB.NS', 'GLENMARK.NS',
    ],
    'Nifty Energy': [
        'RELIANCE.NS', 'ONGC.NS', 'NTPC.NS', 'POWERGRID.NS',
        'BPCL.NS', 'IOC.NS', 'GAIL.NS', 'ADANIGREEN.NS',
        'ADANIPORTS.NS', 'TATAPOWER.NS', 'CESC.NS',
    ],
    'Nifty Auto': [
        'MARUTI.NS', 'TATAMOTORS.NS', 'M&M.NS', 'BAJAJ-AUTO.NS',
        'HEROMOTOCO.NS', 'EICHERMOT.NS', 'TVSMOTORS.NS', 'ASHOKLEY.NS',
        'BALKRISIND.NS', 'MOTHERSON.NS', 'BOSCHLTD.NS',
    ],
    'Nifty Financial Services': [
        'HDFCBANK.NS', 'ICICIBANK.NS', 'SBIN.NS', 'KOTAKBANK.NS',
        'AXISBANK.NS', 'BAJFINANCE.NS', 'BAJAJFINSV.NS', 'HDFCLIFE.NS',
        'SBILIFE.NS', 'ICICIGI.NS', 'MUTHOOTFIN.NS',
    ],
    'Nifty Metal': [
        'TATASTEEL.NS', 'JSWSTEEL.NS', 'HINDALCO.NS', 'SAIL.NS',
        'VEDL.NS', 'COALINDIA.NS', 'NMDC.NS', 'JINDALSTEL.NS',
        'APLAPOLLO.NS', 'RATNAMANI.NS',
    ],
    'Nifty Media': [
        'ZEEL.NS', 'SUNTV.NS', 'NETWORK18.NS', 'PVRINOX.NS',
        'DISHTV.NS', 'NAZARA.NS',
    ],
}

# Factor thresholds
MIN_PARTICIPATION    = 0.70   # Factor 1: at least 70% of up-years stock also up
BETA_MIN             = 0.8    # Factor 2: minimum beta (don't want laggards)
BETA_MAX             = 2.0    # Factor 2: maximum beta (avoid extreme volatility)
MIN_AVG_DAILY_VALUE  = 50     # Factor 3: minimum avg daily traded value in Cr

# Data
LOOKBACK_YEARS = 10
END_DATE   = datetime.today().strftime('%Y-%m-%d')
START_DATE = (datetime.today() - timedelta(days=365 * LOOKBACK_YEARS)).strftime('%Y-%m-%d')

OUTPUT_DIR  = 'output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Anomaly windows configured: {len(ANOMALY_WINDOWS)}')
print(f'Data range: {START_DATE} → {END_DATE}')

Anomaly windows configured: 4
Data range: 2016-06-02 → 2026-05-31


In [4]:
import requests
import pandas as pd

url = "https://www.nseindia.com/api/equity-stock-indices?index=NIFTY%20ENERGY"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)

jsonData = response.json()['data']

for d in jsonData:
    print(d['symbol'])


NIFTY ENERGY
RELIANCE
COALINDIA
ATGL
POWERGRID
ONGC
ADANIPOWER
JPPOWER
POWERINDIA
SUZLON
GVT&D
ENRIN
THERMAX
CGPOWER
ABB
BPCL
BHEL
SIEMENS
JSWENERGY
NTPC
ADANIGREEN
ADANIENSOL
IOC
HINDPETRO
NHPC
OIL
AEGISLOG
GAIL
NLCINDIA
TATAPOWER
PETRONET
TORNTPOWER
INOXWIND
RPOWER
NTPCGREEN
MGL
SJVN
CESC
IGL
CASTROLIND
NAVA


In [4]:
# ── Helper: Download stock data ───────────────────────────────────────────────

def download_stock(ticker):
    """
    Download OHLCV data for a single stock ticker.
    Returns DataFrame or None on failure.
    """
    try:
        raw = yf.download(ticker, start=START_DATE, end=END_DATE,
                          progress=False, auto_adjust=True)
        if raw.empty:
            return None
        if isinstance(raw.columns, pd.MultiIndex):
            raw.columns = raw.columns.get_level_values(0)
        df = raw[['Open', 'High', 'Low', 'Close', 'Volume']].copy()
        df.index.name = 'Date'
        df = df.ffill(limit=3).dropna()
        return df
    except Exception:
        return None


def load_sector_data(sector_name):
    """
    Load sector index data saved by Notebook 1.
    """
    filename = sector_name.replace(' ', '_').lower() + '.csv'
    filepath = os.path.join('data', 'raw', filename)
    if not os.path.exists(filepath):
        print(f'  ⚠ Sector file not found: {filepath}')
        return None
    df = pd.read_csv(filepath, index_col='Date', parse_dates=True)
    return df.sort_index()[['Close']]


def find_nearest_trading_day(df_index, year, month, day):
    """Find the first trading day on or after (year, month, day)."""
    try:
        target = pd.Timestamp(year=year, month=month, day=day)
    except ValueError:
        return None
    candidates = df_index[(df_index >= target) & (df_index.year == year)]
    return candidates[0] if len(candidates) > 0 else None


print('Helper functions defined.')

Helper functions defined.


In [5]:
# ── FACTOR 1 & 2: Participation Rate + Window Beta ────────────────────────────
#
# For each anomaly window:
#   - Find every historical instance of the window (one per year)
#   - Record sector return for that window
#   - Record stock return for the same window
#   - Participation = % of sector-up years where stock was also up
#   - Beta = avg(stock_return) / avg(sector_return) during window years

def compute_participation_and_beta(sector_df, stock_df, window):
    """
    Returns participation rate (0-1) and window beta for a stock
    given a sector DataFrame and window definition.
    """
    s_month = window['start_month']
    s_day   = window['start_day']
    w_days  = window['window_days']

    years = sorted(sector_df.index.year.unique())

    sector_returns = []
    stock_returns  = []

    for year in years:
        # Find entry date in sector
        entry_date = find_nearest_trading_day(sector_df.index, year, s_month, s_day)
        if entry_date is None:
            continue

        # Find exit date: entry + w_days trading days
        entry_pos = sector_df.index.get_loc(entry_date)
        exit_pos  = entry_pos + w_days
        if exit_pos >= len(sector_df):
            continue
        exit_date = sector_df.index[exit_pos]

        # Sector return
        sec_entry = sector_df['Close'].iloc[entry_pos]
        sec_exit  = sector_df['Close'].iloc[exit_pos]
        sec_ret   = (sec_exit - sec_entry) / sec_entry * 100

        # Stock return for same calendar dates
        stk_window = stock_df[(stock_df.index >= entry_date) & (stock_df.index <= exit_date)]
        if len(stk_window) < 2:
            continue
        stk_ret = (stk_window['Close'].iloc[-1] - stk_window['Close'].iloc[0]) / stk_window['Close'].iloc[0] * 100

        sector_returns.append(sec_ret)
        stock_returns.append(stk_ret)

    if len(sector_returns) < 4:
        return None, None

    sector_returns = np.array(sector_returns)
    stock_returns  = np.array(stock_returns)

    # Participation: when sector was up, was stock also up?
    sector_up_idx   = sector_returns > 0
    if sector_up_idx.sum() == 0:
        return None, None
    participation   = (stock_returns[sector_up_idx] > 0).mean()

    # Window beta: ratio of average returns
    avg_sec = np.mean(sector_returns)
    avg_stk = np.mean(stock_returns)
    beta    = avg_stk / avg_sec if avg_sec != 0 else np.nan

    return round(participation, 3), round(beta, 3)


print('Factor 1 & 2 function defined.')

Factor 1 & 2 function defined.


In [6]:
# ── FACTOR 3: Liquidity ───────────────────────────────────────────────────────
#
# Average daily traded value in Crores over the last 60 trading days.
# Value = Close × Volume / 1e7  (converting to Crores)

def compute_liquidity(stock_df, lookback_days=60):
    """
    Returns average daily traded value in Crores over last `lookback_days` trading days.
    """
    recent = stock_df.tail(lookback_days).copy()
    if len(recent) < 10:
        return 0
    # Traded value = Close * Volume (in ₹), convert to Cr (divide by 1e7)
    recent['traded_value_cr'] = recent['Close'] * recent['Volume'] / 1e7
    return round(recent['traded_value_cr'].mean(), 2)


print('Factor 3 function defined.')

Factor 3 function defined.


In [7]:
# ── FACTOR 4: Events Inside Window ───────────────────────────────────────────
#
# Checks for known scheduled event types that typically fall inside the window.
# This is a flag — not a disqualifier by itself, but a warning to check before trading.
#
# Events checked:
#   - Quarterly results season (Q1: Jul-Aug, Q2: Oct-Nov, Q3: Jan-Feb, Q4: Apr-May)
#   - RBI policy meetings (roughly Feb, Apr, Jun, Aug, Oct, Dec)
#   - Union Budget (Feb 1)
#
# Note: For FDA dates (Pharma), you should manually verify PDUFA dates
# on https://www.fda.gov/patients/drug-approvals-and-databases/drug-approvals closer to the window.

RBI_MONTHS   = [2, 4, 6, 8, 10, 12]    # Months when RBI typically meets
RESULTS_WINDOWS = {
    'Q1 results': (7, 15, 8, 31),    # Jul 15 – Aug 31
    'Q2 results': (10, 10, 11, 30),  # Oct 10 – Nov 30
    'Q3 results': (1, 10, 2, 28),    # Jan 10 – Feb 28
    'Q4 results': (4, 10, 5, 31),    # Apr 10 – May 31
}

def check_events_in_window(window):
    """
    Returns a list of event strings that fall inside the anomaly window.
    Uses a reference year (2024) for date math.
    """
    ref_year = 2024
    try:
        w_start = pd.Timestamp(year=ref_year, month=window['start_month'], day=window['start_day'])
    except ValueError:
        return []
    w_end = w_start + pd.Timedelta(days=window['window_days'])

    events = []

    # Results seasons
    for name, (sm, sd, em, ed) in RESULTS_WINDOWS.items():
        try:
            r_start = pd.Timestamp(year=ref_year, month=sm, day=sd)
            r_end   = pd.Timestamp(year=ref_year, month=em, day=ed)
        except ValueError:
            continue
        # Check overlap
        if w_start <= r_end and w_end >= r_start:
            events.append(name)

    # RBI meetings
    for month in RBI_MONTHS:
        rbi_approx = pd.Timestamp(year=ref_year, month=month, day=7)
        if w_start <= rbi_approx <= w_end:
            events.append(f'RBI meeting (~{rbi_approx.strftime("%b %d")})')

    # Budget
    budget_day = pd.Timestamp(year=ref_year, month=2, day=1)
    if w_start <= budget_day <= w_end:
        events.append('Union Budget (Feb 1)')

    return events


print('Factor 4 function defined.')

Factor 4 function defined.


In [8]:
# ── FACTOR 5: Relative Strength at Entry ─────────────────────────────────────
#
# Computed on the most recent data (as of today), not historically.
# This factor is re-checked each year just before the window opens.
#
# Three sub-checks:
#   (a) Is stock above its 20-day moving average? (trend with you)
#   (b) Has stock NOT run >10% in the past 14 days? (avoid chasing)
#   (c) Is stock's 14-day return > sector's 14-day return? (leading the sector)

def compute_relative_strength(stock_df, sector_df):
    """
    Returns a dict of current relative strength indicators.
    """
    if len(stock_df) < 25:
        return {'above_20ma': None, 'recent_run_pct': None, 'leading_sector': None, 'rs_score': 0}

    close = stock_df['Close']

    # (a) Above 20-day MA
    ma20        = close.rolling(20).mean().iloc[-1]
    current     = close.iloc[-1]
    above_20ma  = bool(current > ma20)

    # (b) Recent run — last 14 trading days
    price_14d_ago = close.iloc[-14] if len(close) >= 14 else close.iloc[0]
    recent_run    = (current - price_14d_ago) / price_14d_ago * 100
    not_extended  = recent_run < 10.0

    # (c) Leading sector — compare stock's 14d return to sector's 14d return
    sec_close     = sector_df['Close']
    sec_14d_ago   = sec_close.iloc[-14] if len(sec_close) >= 14 else sec_close.iloc[0]
    sec_run       = (sec_close.iloc[-1] - sec_14d_ago) / sec_14d_ago * 100
    leading       = bool(recent_run > sec_run)

    # RS score: 0-3 (one point per sub-check passed)
    rs_score = int(above_20ma) + int(not_extended) + int(leading)

    return {
        'above_20ma'    : above_20ma,
        'recent_run_pct': round(recent_run, 2),
        'leading_sector': leading,
        'rs_score'      : rs_score,       # 0 = worst, 3 = best
    }


print('Factor 5 function defined.')

Factor 5 function defined.


In [9]:
# ── FACTOR 6: Fundamental Quality Floor ──────────────────────────────────────
#
# Pass/fail only. Fetches from yfinance's info dict.
# Checks:
#   (a) Debt-to-equity < 1.5
#   (b) Promoter holding > 45%  (not directly in yfinance — flagged as manual check)
#   (c) No recent major regulatory action (manual check — flagged)
#
# yfinance provides D/E and market cap reliably.
# Promoter holding requires NSE/BSE shareholding data — flagged for manual review.

def compute_fundamentals(ticker):
    """
    Returns fundamental quality flags for a ticker.
    """
    try:
        info = yf.Ticker(ticker).info

        # Debt-to-equity (yfinance key: debtToEquity — reported as %)
        de_raw = info.get('debtToEquity', None)
        if de_raw is not None:
            de_ratio     = de_raw / 100   # yfinance reports D/E * 100
            de_pass      = de_ratio < 1.5
            de_display   = round(de_ratio, 2)
        else:
            de_ratio     = None
            de_pass      = None           # unknown
            de_display   = 'N/A'

        # Market cap (for context)
        mkt_cap_cr = round(info.get('marketCap', 0) / 1e7, 0)  # convert to Cr

        # Trailing P/E
        pe = info.get('trailingPE', None)
        pe = round(pe, 1) if pe else 'N/A'

        return {
            'de_ratio'           : de_display,
            'de_pass'            : de_pass,
            'mkt_cap_cr'         : mkt_cap_cr,
            'trailing_pe'        : pe,
            'promoter_holding'   : 'Manual check',   # requires BSE/NSE data
            'regulatory_flag'    : 'Manual check',   # requires news review
        }

    except Exception as e:
        return {
            'de_ratio'           : 'Error',
            'de_pass'            : None,
            'mkt_cap_cr'         : None,
            'trailing_pe'        : None,
            'promoter_holding'   : 'Manual check',
            'regulatory_flag'    : 'Manual check',
        }


print('Factor 6 function defined.')

Factor 6 function defined.


In [10]:
# ── Main Loop: Run All Factors for All Windows ────────────────────────────────
# This is the heavy cell — downloads stock data and computes all 6 factors.
# Expect 10–20 minutes depending on number of stocks and internet speed.

all_results = []

for window in ANOMALY_WINDOWS:
    print(f"\n{'═'*60}")
    print(f"Window: {window['name']}")
    print(f"{'═'*60}")

    sector_name   = window['sector']
    stock_tickers = SECTOR_STOCKS.get(sector_name, [])

    if not stock_tickers:
        print(f'  No stocks configured for {sector_name}')
        continue

    # Load sector index data
    sector_df = load_sector_data(sector_name)
    if sector_df is None:
        print(f'  Could not load sector data for {sector_name}')
        continue

    # Factor 4: events in this window (same for all stocks in window)
    events_in_window = check_events_in_window(window)
    events_str       = ', '.join(events_in_window) if events_in_window else 'None'
    print(f'  Events in window: {events_str}')

    for ticker in tqdm(stock_tickers, desc=f'  Analyzing {sector_name}'):
        # Download stock data
        stock_df = download_stock(ticker)
        if stock_df is None or len(stock_df) < 100:
            print(f'    ⚠ Skipping {ticker} — insufficient data')
            continue

        # Factor 1 & 2
        participation, beta = compute_participation_and_beta(sector_df, stock_df, window)
        if participation is None:
            continue

        # Factor 3
        liquidity_cr = compute_liquidity(stock_df)

        # Factor 5
        rs = compute_relative_strength(stock_df, sector_df)

        # Factor 6
        fundamentals = compute_fundamentals(ticker)

        # Compute filter flags
        f1_pass = participation >= MIN_PARTICIPATION
        f2_pass = BETA_MIN <= beta <= BETA_MAX if beta is not None else False
        f3_pass = liquidity_cr >= MIN_AVG_DAILY_VALUE
        f6_pass = fundamentals['de_pass']  # None = unknown, True = pass, False = fail

        # Overall pre-computed verdict (factors 1,2,3,6)
        hard_fails = [not f1_pass, not f2_pass, not f3_pass]
        if f6_pass is False:
            hard_fails.append(True)
        pre_verdict = 'PASS' if not any(hard_fails) else 'FILTER OUT'

        all_results.append({
            # Identity
            'Window'              : window['name'],
            'Sector'              : sector_name,
            'Ticker'              : ticker,
            # Factor 1
            'Participation Rate'  : participation,
            'F1 Pass'             : '✓' if f1_pass else '✗',
            # Factor 2
            'Window Beta'         : beta,
            'F2 Pass'             : '✓' if f2_pass else '✗',
            # Factor 3
            'Avg Daily Value (Cr)': liquidity_cr,
            'F3 Pass'             : '✓' if f3_pass else '✗',
            # Factor 4
            'Events in Window'    : events_str,
            # Factor 5 (current)
            'Above 20MA'          : rs['above_20ma'],
            'Recent Run %'        : rs['recent_run_pct'],
            'Leading Sector'      : rs['leading_sector'],
            'RS Score (0-3)'      : rs['rs_score'],
            # Factor 6
            'D/E Ratio'           : fundamentals['de_ratio'],
            'F6 D/E Pass'         : '✓' if f6_pass else ('?' if f6_pass is None else '✗'),
            'Mkt Cap (Cr)'        : fundamentals['mkt_cap_cr'],
            'Trailing P/E'        : fundamentals['trailing_pe'],
            'Promoter Holding'    : fundamentals['promoter_holding'],
            'Regulatory Flag'     : fundamentals['regulatory_flag'],
            # Verdict
            'Pre-screen Verdict'  : pre_verdict,
        })

print(f'\n── Analysis complete ──')
print(f'Total stock-window combinations: {len(all_results)}')


════════════════════════════════════════════════════════════
Window: FMCG Mar-May Rally
════════════════════════════════════════════════════════════
  Events in window: Q4 results, RBI meeting (~Apr 07)


  Analyzing Nifty FMCG:   0%|          | 0/12 [00:00<?, ?it/s]

$MCDOWELL-N.NS: possibly delisted; no timezone found

1 Failed download:
['MCDOWELL-N.NS']: possibly delisted; no timezone found


    ⚠ Skipping MCDOWELL-N.NS — insufficient data

════════════════════════════════════════════════════════════
Window: Pharma Aug-Sep Rally
════════════════════════════════════════════════════════════
  Events in window: Q1 results


  Analyzing Nifty Pharma:   0%|          | 0/12 [00:00<?, ?it/s]


════════════════════════════════════════════════════════════
Window: Energy Aug Mid Rally
════════════════════════════════════════════════════════════
  Events in window: Q1 results


  Analyzing Nifty Energy:   0%|          | 0/11 [00:00<?, ?it/s]


════════════════════════════════════════════════════════════
Window: Pharma Aug Extended
════════════════════════════════════════════════════════════
  Events in window: Q1 results


  Analyzing Nifty Pharma:   0%|          | 0/12 [00:00<?, ?it/s]


── Analysis complete ──
Total stock-window combinations: 46


In [12]:
# ── Build Results DataFrame & Preview ────────────────────────────────────────

results_df = pd.DataFrame(all_results)

# Sort: PASS first, then by participation rate desc, then beta desc
results_df['_sort_verdict'] = (results_df['Pre-screen Verdict'] == 'PASS').astype(int)
results_df = results_df.sort_values(
    ['Window', '_sort_verdict', 'Participation Rate', 'Window Beta'],
    ascending=[True, False, False, False]
).drop(columns='_sort_verdict').reset_index(drop=True)

print('\nSample results (first 15 rows):')
preview_cols = ['Window', 'Ticker', 'Participation Rate', 'Window Beta',
                'Avg Daily Value (Cr)', 'RS Score (0-3)', 'Pre-screen Verdict']
print(results_df[preview_cols].head(15).to_string(index=False))

print(f"\n── Passed pre-screen: {(results_df['Pre-screen Verdict']=='PASS').sum()} stocks")
print(f"── Filtered out    : {(results_df['Pre-screen Verdict']=='FILTER OUT').sum()} stocks")


Sample results (first 15 rows):
              Window        Ticker  Participation Rate  Window Beta  Avg Daily Value (Cr)  RS Score (0-3) Pre-screen Verdict
Energy Aug Mid Rally ADANIPORTS.NS               0.857        1.484                501.19               3               PASS
Energy Aug Mid Rally       GAIL.NS               0.857        1.155                202.46               2               PASS
Energy Aug Mid Rally       NTPC.NS               0.714        1.787                466.00               1               PASS
Energy Aug Mid Rally       BPCL.NS               0.714        1.505                406.60               2               PASS
Energy Aug Mid Rally        IOC.NS               0.714        0.949                294.49               1               PASS
Energy Aug Mid Rally       CESC.NS               1.000        3.545                 58.88               1         FILTER OUT
Energy Aug Mid Rally       ONGC.NS               0.857        0.291                611.05   

In [13]:
# ── Export to Excel ───────────────────────────────────────────────────────────

OUTPUT_PATH = os.path.join(OUTPUT_DIR, 'stock_selection.xlsx')

with pd.ExcelWriter(OUTPUT_PATH, engine='openpyxl') as writer:
    # One sheet per anomaly window
    for window in ANOMALY_WINDOWS:
        wdf = results_df[results_df['Window'] == window['name']].copy()
        sheet_name = window['name'][:31]  # Excel sheet name max 31 chars
        wdf.to_excel(writer, sheet_name=sheet_name, index=False)

    # Combined sheet — all PASS stocks across all windows
    passed = results_df[results_df['Pre-screen Verdict'] == 'PASS'].copy()
    passed.to_excel(writer, sheet_name='All Passed Stocks', index=False)

print('Applying formatting...')

wb = load_workbook(OUTPUT_PATH)

PASS_FILL   = PatternFill('solid', fgColor='C6EFCE')
FAIL_FILL   = PatternFill('solid', fgColor='FFC7CE')
WARN_FILL   = PatternFill('solid', fgColor='FFEB9C')
HEADER_FILL = PatternFill('solid', fgColor='1F3864')
ALT_FILL    = PatternFill('solid', fgColor='EEF2FF')
HEADER_FONT = Font(bold=True, color='FFFFFF', size=10)
THIN_BORDER = Border(bottom=Side(style='thin', color='CCCCCC'))

VERDICT_COL_NAME = 'Pre-screen Verdict'

for sheet_name in wb.sheetnames:
    ws = wb[sheet_name]

    # Header
    for cell in ws[1]:
        cell.fill      = HEADER_FILL
        cell.font      = HEADER_FONT
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        cell.border    = THIN_BORDER
    ws.row_dimensions[1].height = 30
    ws.freeze_panes = 'A2'

    # Find column indices
    headers = [cell.value for cell in ws[1]]
    def col_idx(name):
        return headers.index(name) + 1 if name in headers else None

    verdict_col = col_idx('Pre-screen Verdict')
    f1_col      = col_idx('F1 Pass')
    f2_col      = col_idx('F2 Pass')
    f3_col      = col_idx('F3 Pass')
    f6_col      = col_idx('F6 D/E Pass')
    rs_col      = col_idx('RS Score (0-3)')

    # Data rows
    for row_idx, row in enumerate(ws.iter_rows(min_row=2), start=2):
        base_fill = ALT_FILL if row_idx % 2 == 0 else PatternFill()
        for cell in row:
            cell.alignment = Alignment(horizontal='center', vertical='center')
            cell.border    = THIN_BORDER
            cell.fill      = base_fill
        ws.row_dimensions[row_idx].height = 15

        # Verdict column color
        if verdict_col:
            v_cell = row[verdict_col - 1]
            if v_cell.value == 'PASS':
                v_cell.fill = PASS_FILL
                v_cell.font = Font(bold=True, color='375623')
            elif v_cell.value == 'FILTER OUT':
                v_cell.fill = FAIL_FILL
                v_cell.font = Font(bold=True, color='9C0006')

        # Pass/fail columns
        for c in [f1_col, f2_col, f3_col, f6_col]:
            if c:
                cell = row[c - 1]
                if cell.value == '✓':
                    cell.fill = PASS_FILL
                elif cell.value == '✗':
                    cell.fill = FAIL_FILL
                elif cell.value == '?':
                    cell.fill = WARN_FILL

        # RS Score color
        if rs_col:
            rs_cell = row[rs_col - 1]
            try:
                rs_val = int(rs_cell.value)
                if rs_val == 3:
                    rs_cell.fill = PASS_FILL
                elif rs_val == 2:
                    rs_cell.fill = WARN_FILL
                else:
                    rs_cell.fill = FAIL_FILL
            except (TypeError, ValueError):
                pass

    # Auto column widths
    for col in ws.columns:
        max_len    = max(len(str(cell.value or '')) for cell in col)
        col_letter = get_column_letter(col[0].column)
        ws.column_dimensions[col_letter].width = min(max(max_len + 3, 10), 28)

wb.save(OUTPUT_PATH)
print(f'\n✅ Stock selection report saved: {OUTPUT_PATH}')
print(f'   Sheets: one per anomaly window + "All Passed Stocks" summary')

Applying formatting...

✅ Stock selection report saved: output\stock_selection.xlsx
   Sheets: one per anomaly window + "All Passed Stocks" summary


In [14]:
# ── Final Readable Summary ────────────────────────────────────────────────────
# Clean console output of top candidates per window

print('═' * 65)
print('    TOP STOCK CANDIDATES PER ANOMALY WINDOW')
print('═' * 65)

for window in ANOMALY_WINDOWS:
    wname = window['name']
    wdf   = results_df[
        (results_df['Window'] == wname) &
        (results_df['Pre-screen Verdict'] == 'PASS')
    ].head(5)

    print(f'\n{wname}')
    print(f"  Events: {wdf['Events in Window'].iloc[0] if len(wdf) > 0 else 'N/A'}")
    print(f"  {'Ticker':<18} {'Particip':>8} {'Beta':>6} {'Liq(Cr)':>9} {'RS':>4} {'D/E':>6}")
    print(f"  {'-'*55}")

    if len(wdf) == 0:
        print('  No stocks passed all filters for this window.')
        print('  → Try lowering MIN_PARTICIPATION or MIN_AVG_DAILY_VALUE in config.')
    else:
        for _, row in wdf.iterrows():
            print(
                f"  {row['Ticker']:<18}"
                f" {row['Participation Rate']:>8.0%}"
                f" {row['Window Beta']:>6.2f}"
                f" {row['Avg Daily Value (Cr)']:>9.0f}"
                f" {row['RS Score (0-3)']:>4}"
                f" {str(row['D/E Ratio']):>6}"
            )

print('\n═' * 65)
print('Note: Factor 4 (events) shown above — verify manually before trading.')
print('Note: Factor 5 (RS score) reflects current data — re-run on entry day.')
print('Note: Promoter holding & regulatory flag require manual BSE/NSE check.')
print(f'\nOpen output/stock_selection.xlsx for the full formatted report.')

═════════════════════════════════════════════════════════════════
    TOP STOCK CANDIDATES PER ANOMALY WINDOW
═════════════════════════════════════════════════════════════════

FMCG Mar-May Rally
  Events: Q4 results, RBI meeting (~Apr 07)
  Ticker             Particip   Beta   Liq(Cr)   RS    D/E
  -------------------------------------------------------
  MARICO.NS              100%   1.87       145    3   0.12
  TATACONSUM.NS          100%   1.80       215    3   0.12
  BRITANNIA.NS            90%   1.59       248    1   0.27
  GODREJCP.NS             90%   1.46       182    1   0.35
  HINDUNILVR.NS           90%   1.27       436    1   0.03

Pharma Aug-Sep Rally
  Events: Q1 results
  Ticker             Particip   Beta   Liq(Cr)   RS    D/E
  -------------------------------------------------------
  LUPIN.NS                90%   1.93       278    1   0.29
  DIVISLAB.NS             80%   1.72       194    2    0.0
  ALKEM.NS                80%   1.63        61    1   0.17
  GLENMARK.